In [1]:
import os, sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine, make_url, text

load_dotenv(find_dotenv(usecwd=True))
url = make_url(os.environ["DATABASE_URL"])
os.environ.update({
    "DBT_HOST": url.host, "DBT_PORT": str(url.port or 5432),
    "DBT_USER": url.username, "DBT_PASSWORD": url.password,
    "DBT_DBNAME": url.database, "DBT_USE_COLORS": "false",
})
engine = create_engine(os.environ["DATABASE_URL"])

PROJECT_DIR = Path.cwd().parent
DBT_DIR = PROJECT_DIR / "fundlens_dbt"
for sub in ["models/intermediate", "models/marts"]:
    (DBT_DIR / sub).mkdir(parents=True, exist_ok=True)

exe_dir = Path(sys.executable).parent
DBT_EXE = exe_dir / "dbt.exe" if (exe_dir / "dbt.exe").exists() else exe_dir / "Scripts" / "dbt.exe"
DBT = f"--project-dir {DBT_DIR} --profiles-dir {DBT_DIR}"
print(sys.executable)

C:\Users\chetanksoftom\AppData\Local\Programs\Python\Python311\python.exe


In [2]:
%%writefile ../fundlens_dbt/dbt_project.yml
name: fundlens
version: "1.0.0"
profile: fundlens

model-paths: ["models"]
snapshot-paths: ["snapshots"]
test-paths: ["tests"]
macro-paths: ["macros"]

vars:
  risk_free_rate: 0.065     # yearly risk-free rate used for Sharpe ratio
  spike_threshold: 0.15     # a 15%+ jump that reverses next day = bad data

models:
  fundlens:
    staging:
      +schema: staging
      +materialized: view
    core:
      +schema: core
      +materialized: table
    intermediate:
      +schema: intermediate
      +materialized: table
    marts:
      +schema: marts
      +materialized: table

Overwriting ../fundlens_dbt/dbt_project.yml


In [3]:
%%writefile ../fundlens_dbt/models/staging/_sources.yml
version: 2

sources:
  - name: raw
    schema: raw
    tables:
      - name: amfi_nav_daily
        description: Daily NAV file from AMFI, all schemes, one row per scheme per day
        config:
          loaded_at_field: loaded_at
          freshness:
            warn_after: {count: 2, period: day}
            error_after: {count: 5, period: day}
      - name: nav_history
        description: Historical NAVs from mfapi.in for equity growth schemes
      - name: scheme_meta
        description: Scheme details from mfapi.in
      - name: backfill_status
        description: One row per scheme showing if its history download worked

Overwriting ../fundlens_dbt/models/staging/_sources.yml


In [4]:
%%writefile ../fundlens_dbt/models/intermediate/int_nav_flagged.sql
-- marks one-day spikes: a big jump that fully reverses the next day (bad data, not real moves)
{{ config(materialized='view') }}

with base as (
    select
        scheme_code,
        nav_date,
        nav,
        lag(nav)  over w as prev_nav,
        lead(nav) over w as next_nav
    from {{ ref('fct_nav_daily') }}
    window w as (partition by scheme_code order by nav_date)
)

select
    scheme_code,
    nav_date,
    nav,
    prev_nav,
    coalesce(
        abs(nav / prev_nav - 1) > {{ var('spike_threshold') }}
        and abs(next_nav / nav - 1) > {{ var('spike_threshold') }}
        and sign(nav / prev_nav - 1) <> sign(next_nav / nav - 1),
        false
    ) as is_spike
from base

Writing ../fundlens_dbt/models/intermediate/int_nav_flagged.sql


In [5]:
%%writefile ../fundlens_dbt/models/intermediate/int_nav_clean.sql
-- clean NAV series (spikes removed) with daily returns
{{ config(indexes=[{'columns': ['scheme_code', 'nav_date'], 'unique': True}]) }}

with clean as (
    select scheme_code, nav_date, nav
    from {{ ref('int_nav_flagged') }}
    where not is_spike
)

select
    scheme_code,
    nav_date,
    nav,
    lag(nav_date) over w                as prev_date,
    nav / lag(nav) over w - 1           as daily_return,
    ln(nav / lag(nav) over w)           as log_return
from clean
window w as (partition by scheme_code order by nav_date)

Writing ../fundlens_dbt/models/intermediate/int_nav_clean.sql


In [6]:
%%writefile ../fundlens_dbt/models/marts/mart_fund_metrics.sql
-- one row per scheme: returns, risk and rank inside its category
with latest as (
    select scheme_code, max(nav_date) as as_of
    from {{ ref('int_nav_clean') }}
    group by 1
),

points as (
    select
        l.scheme_code,
        l.as_of,
        n0.nav as nav_latest,
        n1.nav_date as date_1y, n1.nav as nav_1y,
        n3.nav_date as date_3y, n3.nav as nav_3y,
        n5.nav_date as date_5y, n5.nav as nav_5y
    from latest l
    join {{ ref('int_nav_clean') }} n0
      on n0.scheme_code = l.scheme_code and n0.nav_date = l.as_of
    {% for yrs in [1, 3, 5] %}
    left join lateral (
        select nav_date, nav
        from {{ ref('int_nav_clean') }} c
        where c.scheme_code = l.scheme_code
          and c.nav_date <= (l.as_of - interval '{{ yrs }} year')::date
          and c.nav_date >  (l.as_of - interval '{{ yrs }} year' - interval '10 day')::date
        order by c.nav_date desc
        limit 1
    ) n{{ yrs }} on true
    {% endfor %}
),

risk as (
    select
        c.scheme_code,
        stddev_samp(c.log_return) filter (where c.nav_date > (l.as_of - interval '1 year')::date)  * sqrt(252) as volatility_1y,
        stddev_samp(c.log_return) filter (where c.nav_date > (l.as_of - interval '3 year')::date)  * sqrt(252) as volatility_3y,
        stddev_samp(c.log_return) filter (where c.nav_date > (l.as_of - interval '3 year')::date and c.log_return < 0)
            * sqrt(252) as downside_vol_3y
    from {{ ref('int_nav_clean') }} c
    join latest l using (scheme_code)
    group by 1
),

drawdown as (
    select scheme_code, min(nav / running_max - 1) as max_drawdown_3y
    from (
        select
            c.scheme_code,
            c.nav,
            max(c.nav) over (partition by c.scheme_code order by c.nav_date) as running_max
        from {{ ref('int_nav_clean') }} c
        join latest l using (scheme_code)
        where c.nav_date >= (l.as_of - interval '3 year')::date
    ) x
    group by 1
),

metrics as (
    select
        p.scheme_code,
        p.as_of,
        p.nav_latest,
        (power(p.nav_latest / p.nav_1y, 365.0 / (p.as_of - p.date_1y)) - 1)::numeric as return_1y,
        (power(p.nav_latest / p.nav_3y, 365.0 / (p.as_of - p.date_3y)) - 1)::numeric as cagr_3y,
        (power(p.nav_latest / p.nav_5y, 365.0 / (p.as_of - p.date_5y)) - 1)::numeric as cagr_5y,
        r.volatility_1y::numeric   as volatility_1y,
        r.volatility_3y::numeric   as volatility_3y,
        r.downside_vol_3y::numeric as downside_vol_3y,
        d.max_drawdown_3y::numeric as max_drawdown_3y
    from points p
    left join risk r using (scheme_code)
    left join drawdown d using (scheme_code)
),

final as (
    select
        m.*,
        (m.cagr_3y - {{ var('risk_free_rate') }}) / nullif(m.volatility_3y, 0)    as sharpe_3y,
        (m.cagr_3y - {{ var('risk_free_rate') }}) / nullif(m.downside_vol_3y, 0)  as sortino_3y,
        s.scheme_name,
        s.fund_house,
        s.asset_class,
        s.sub_category,
        s.plan_type,
        m.as_of >= (select max(as_of) from latest) - 7 as is_active
    from metrics m
    join {{ ref('dim_scheme') }} s
      on s.scheme_code = m.scheme_code and s.is_current
)

select
    *,
    case when is_active and cagr_3y is not null then
        rank() over (partition by is_active and cagr_3y is not null, sub_category, plan_type
                     order by cagr_3y desc)
    end as rank_cagr_3y_in_category,
    case when is_active and sharpe_3y is not null then
        rank() over (partition by is_active and sharpe_3y is not null, sub_category, plan_type
                     order by sharpe_3y desc)
    end as rank_sharpe_3y_in_category,
    count(*) filter (where is_active and cagr_3y is not null)
        over (partition by sub_category, plan_type) as funds_in_category
from final

Writing ../fundlens_dbt/models/marts/mart_fund_metrics.sql


In [7]:
%%writefile ../fundlens_dbt/models/marts/mart_rolling_returns.sql
-- month-end NAV per scheme with 1-year and 3-year rolling returns (for consistency charts)
with monthly as (
    select distinct on (scheme_code, date_trunc('month', nav_date))
        scheme_code,
        date_trunc('month', nav_date)::date as month,
        nav_date,
        nav
    from {{ ref('int_nav_clean') }}
    order by scheme_code, date_trunc('month', nav_date), nav_date desc
),

lagged as (
    select
        *,
        lag(nav, 12)   over w as nav_12m,
        lag(month, 12) over w as month_12m,
        lag(nav, 36)   over w as nav_36m,
        lag(month, 36) over w as month_36m
    from monthly
    window w as (partition by scheme_code order by month)
)

select
    l.scheme_code,
    l.month,
    l.nav_date,
    l.nav,
    case when l.month_12m = (l.month - interval '12 month')::date
         then l.nav / l.nav_12m - 1 end                     as rolling_1y_return,
    case when l.month_36m = (l.month - interval '36 month')::date
         then power(l.nav / l.nav_36m, 1.0 / 3) - 1 end     as rolling_3y_cagr,
    s.sub_category,
    s.plan_type
from lagged l
join {{ ref('dim_scheme') }} s
  on s.scheme_code = l.scheme_code and s.is_current

Writing ../fundlens_dbt/models/marts/mart_rolling_returns.sql


In [8]:
%%writefile ../fundlens_dbt/models/marts/mart_rolling_returns.sql
-- month-end NAV per scheme with 1-year and 3-year rolling returns (for consistency charts)
with monthly as (
    select distinct on (scheme_code, date_trunc('month', nav_date))
        scheme_code,
        date_trunc('month', nav_date)::date as month,
        nav_date,
        nav
    from {{ ref('int_nav_clean') }}
    order by scheme_code, date_trunc('month', nav_date), nav_date desc
),

lagged as (
    select
        *,
        lag(nav, 12)   over w as nav_12m,
        lag(month, 12) over w as month_12m,
        lag(nav, 36)   over w as nav_36m,
        lag(month, 36) over w as month_36m
    from monthly
    window w as (partition by scheme_code order by month)
)

select
    l.scheme_code,
    l.month,
    l.nav_date,
    l.nav,
    case when l.month_12m = (l.month - interval '12 month')::date
         then l.nav / l.nav_12m - 1 end                     as rolling_1y_return,
    case when l.month_36m = (l.month - interval '36 month')::date
         then power(l.nav / l.nav_36m, 1.0 / 3) - 1 end     as rolling_3y_cagr,
    s.sub_category,
    s.plan_type
from lagged l
join {{ ref('dim_scheme') }} s
  on s.scheme_code = l.scheme_code and s.is_current

Overwriting ../fundlens_dbt/models/marts/mart_rolling_returns.sql


In [9]:
%%writefile ../fundlens_dbt/models/marts/mart_direct_vs_regular.sql
-- pairs the Direct and Regular plan of the same fund and compares returns.
-- the gap is roughly the extra yearly cost (commission) of the Regular plan.
with named as (
    select
        *,
        trim(regexp_replace(
            regexp_replace(lower(scheme_name), '\m(direct|regular|plan|growth|option)\M', '', 'g'),
            '[^a-z0-9]+', ' ', 'g'
        )) as base_name
    from {{ ref('mart_fund_metrics') }}
    where is_active
),

direct as (
    select distinct on (fund_house, sub_category, base_name) *
    from named where plan_type = 'Direct'
    order by fund_house, sub_category, base_name, scheme_code
),

regular as (
    select distinct on (fund_house, sub_category, base_name) *
    from named where plan_type = 'Regular'
    order by fund_house, sub_category, base_name, scheme_code
)

select
    d.fund_house,
    d.sub_category,
    d.base_name                              as fund_name,
    d.scheme_code                            as direct_code,
    r.scheme_code                            as regular_code,
    d.return_1y                              as direct_return_1y,
    r.return_1y                              as regular_return_1y,
    d.cagr_3y                                as direct_cagr_3y,
    r.cagr_3y                                as regular_cagr_3y,
    d.cagr_5y                                as direct_cagr_5y,
    r.cagr_5y                                as regular_cagr_5y,
    d.return_1y - r.return_1y                as gap_1y,
    d.cagr_3y   - r.cagr_3y                  as gap_3y,
    d.cagr_5y   - r.cagr_5y                  as gap_5y
from direct d
join regular r using (fund_house, sub_category, base_name)

Writing ../fundlens_dbt/models/marts/mart_direct_vs_regular.sql


In [10]:
%%writefile ../fundlens_dbt/models/marts/dq_nav_issues.sql
-- data quality report: one row per problem found in the NAV data
with spikes as (
    select scheme_code, nav_date, 'spike_removed' as issue,
           'NAV ' || nav || ' vs previous ' || prev_nav as detail
    from {{ ref('int_nav_flagged') }}
    where is_spike
),

gaps as (
    select scheme_code, nav_date, 'gap_over_10_days' as issue,
           (nav_date - prev_date) || ' days since previous NAV' as detail
    from {{ ref('int_nav_clean') }}
    where nav_date - prev_date > 10
),

stale as (
    select scheme_code, nav_date, 'stale_nav_5_days' as issue,
           'NAV ' || nav || ' unchanged for 5 records' as detail
    from (
        select *,
               count(*) filter (where daily_return = 0)
                   over (partition by scheme_code order by nav_date rows 3 preceding) as zero_days
        from {{ ref('int_nav_clean') }}
    ) x
    where zero_days = 4
),

big_moves as (
    select scheme_code, nav_date, 'big_move_kept' as issue,
           'daily return ' || round(daily_return * 100, 1) || '%' as detail
    from {{ ref('int_nav_clean') }}
    where abs(daily_return) > {{ var('spike_threshold') }}
),

stopped as (
    select scheme_code, max(nav_date) as nav_date, 'stopped_reporting' as issue,
           'last NAV more than 7 days before the latest date' as detail
    from {{ ref('int_nav_clean') }}
    group by 1
    having max(nav_date) < (select max(nav_date) from {{ ref('int_nav_clean') }}) - 7
)

select * from spikes
union all select * from gaps
union all select * from stale
union all select * from big_moves
union all select * from stopped

Writing ../fundlens_dbt/models/marts/dq_nav_issues.sql


In [11]:
%%writefile ../fundlens_dbt/models/marts/_marts.yml
version: 2

models:
  - name: mart_fund_metrics
    description: One row per scheme with returns, risk, Sharpe and category rank
    columns:
      - name: scheme_code
        data_tests: [unique, not_null]
      - name: plan_type
        data_tests:
          - accepted_values:
              arguments:
                values: ['Direct', 'Regular']
  - name: mart_rolling_returns
    description: Month-end NAV with 1y and 3y rolling returns
  - name: mart_direct_vs_regular
    description: Direct vs Regular plan of the same fund, return gap
    columns:
      - name: direct_code
        data_tests: [unique]
      - name: regular_code
        data_tests: [unique]
  - name: dq_nav_issues
    description: Data quality problems found in NAV data

Writing ../fundlens_dbt/models/marts/_marts.yml


In [12]:
tests = {
    "assert_metrics_in_sane_range.sql": """-- warn if any active fund has unrealistic numbers (usually bad source data)
{{ config(severity='warn') }}
select scheme_code, scheme_name, return_1y, cagr_3y, volatility_3y
from {{ ref('mart_fund_metrics') }}
where is_active
  and (abs(return_1y) > 2 or abs(cagr_3y) > 1 or volatility_3y > 1)""",
    "assert_few_big_moves_left.sql": """-- warn if large daily moves remain after spike cleaning
{{ config(severity='warn') }}
select * from {{ ref('dq_nav_issues') }} where issue = 'big_move_kept'""",
}
for name, sql in tests.items():
    (DBT_DIR / "tests" / name).write_text(sql)
print(sorted(p.name for p in (DBT_DIR / "tests").iterdir()))

['assert_dim_one_current_row.sql', 'assert_fct_one_row_per_scheme_day.sql', 'assert_few_big_moves_left.sql', 'assert_metrics_in_sane_range.sql', 'assert_nav_positive.sql']


In [13]:
!{DBT_EXE} build {DBT}

06:24:13  Running with dbt=1.12.5
06:24:15  Registered adapter: postgres=1.11.0
06:24:15  Unable to do partial parsing because a project config has changed
06:24:18  Found 11 models, 22 data tests, 1 snapshot, 4 sources, 478 macros
06:24:18  
06:24:18  Concurrency: 4 threads (target='dev')
06:24:18  
06:24:21  1 of 34 START sql view model staging.stg_amfi_nav_daily ........................ [RUN]
06:24:21  2 of 34 START sql view model staging.stg_nav_history ........................... [RUN]
06:24:21  2 of 34 OK created sql view model staging.stg_nav_history ...................... [CREATE VIEW in 0.33s]
06:24:21  1 of 34 OK created sql view model staging.stg_amfi_nav_daily ................... [CREATE VIEW in 0.33s]
06:24:21  3 of 34 START test not_null_stg_amfi_nav_daily_nav ............................. [RUN]
06:24:21  4 of 34 START test not_null_stg_amfi_nav_daily_scheme_code ..................... [RUN]
06:24:21  4 of 34 PASS not_null_stg_amfi_nav_daily_scheme_code ...................

In [14]:
!{DBT_EXE} source freshness {DBT}

06:26:12  Running with dbt=1.12.5
06:26:13  Registered adapter: postgres=1.11.0
06:26:14  Found 11 models, 22 data tests, 1 snapshot, 4 sources, 478 macros
06:26:14  
06:26:14  Concurrency: 4 threads (target='dev')
06:26:14  
06:26:14  1 of 1 START freshness of raw.amfi_nav_daily ................................... [RUN]
06:26:15  1 of 1 PASS freshness of raw.amfi_nav_daily .................................... [PASS in 0.17s]
06:26:15  
06:26:15  Finished running 1 source in 0 hours 0 minutes and 1.21 seconds (1.21s).
06:26:15  Done.


In [15]:
print("Data quality summary")
display(pd.read_sql(text("""
    select issue, count(*) as rows, count(distinct scheme_code) as schemes
    from marts.dq_nav_issues group by 1 order by 2 desc
"""), engine))

print("Top 3 Direct funds per category by 3-year CAGR")
display(pd.read_sql(text("""
    select sub_category, rank_cagr_3y_in_category as rank, scheme_name,
           round(cagr_3y * 100, 2) as cagr_3y_pct,
           round(volatility_3y * 100, 2) as vol_3y_pct,
           round(sharpe_3y, 2) as sharpe,
           round(max_drawdown_3y * 100, 2) as max_dd_pct,
           funds_in_category
    from marts.mart_fund_metrics
    where plan_type = 'Direct' and rank_cagr_3y_in_category <= 3
    order by sub_category, rank
"""), engine))

print("Direct vs Regular: average yearly return gap")
display(pd.read_sql(text("""
    select sub_category, count(*) as fund_pairs,
           round(avg(gap_3y) * 100, 2) as avg_gap_3y_pct,
           round(avg(gap_5y) * 100, 2) as avg_gap_5y_pct
    from marts.mart_direct_vs_regular
    where gap_3y is not null
    group by 1 order by fund_pairs desc
"""), engine))

Data quality summary


,issue,rows,schemes
0,big_move_kept,24,24
1,stale_nav_5_days,19,7
2,gap_over_10_days,8,4
3,spike_removed,2,2


Top 3 Direct funds per category by 3-year CAGR


,sub_category,rank,scheme_name,cagr_3y_pct,vol_3y_pct,sharpe,max_dd_pct,funds_in_category
0,Contra Fund,1,Kotak Contra Fund,12.89,14.68,0.44,-18.95,3
1,Contra Fund,2,Invesco India Contra Fund,12.48,15.06,0.40,-18.64,3
2,Contra Fund,3,SBI CONTRA FUND,8.99,12.60,0.20,-15.84,3
3,Dividend Yield Fund,1,LIC MF Dividend Yield Fund,17.01,16.14,0.65,-20.44,9
4,Dividend Yield Fund,2,ICICI Prudential Dividend Yield Fund,12.69,12.96,0.48,-15.88,9
5,Dividend Yield Fund,3,Tata Dividend Yield Fund,12.65,16.18,0.38,-20.44,9
6,ELSS,1,Motilal Oswal ELSS Tax Saver Fund,20.54,19.45,0.72,-27.71,17
7,ELSS,2,ITI ELSS Tax Saver Fund,14.55,17.69,0.46,-22.61,17
8,ELSS,3,Quant ELSS Tax Saver Fund,13.58,16.83,0.42,-25.84,17
9,ELSS- Tax Saver Fund,1,JM ELSS - Tax Saver Fund,14.84,16.31,0.51,-20.85,20


Direct vs Regular: average yearly return gap


,sub_category,fund_pairs,avg_gap_3y_pct,avg_gap_5y_pct
0,Sectoral/ Thematic,100,1.24,1.17
1,Flexi Cap Fund,36,1.26,1.18
2,Large Cap Fund,30,1.18,1.13
3,Thematic Fund,29,1.44,1.41
4,Mid Cap Fund,28,1.26,1.22
5,Focused Fund,27,1.34,1.33
6,Large & Mid Cap Fund,26,1.22,1.25
7,Small Cap Fund,23,0.88,1.03
8,Multi Cap Fund,22,1.37,1.38
9,Value Fund,20,1.19,1.14
